# 36장 · 검색과 답변의 실패를 따로 평가하기

검색 목록에 정답이 있지만 모델 답에는 다른 숫자가 적힐 수 있습니다. 먼저 원문을 찾았는지 계산하고 그다음 답과 인용을 읽습니다. 아래 순위는 지표의 차이를 관찰하기 위해 만든 예입니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 첫 정답이 두 번째에 있을 때

Recall@k는 기대 원문 중 앞 k개에서 찾은 원문의 비율입니다. 첫 정답의 순위를 보려면 순위의 역수(RR)를 구합니다. 여러 답변 가능 질문의 RR을 평균한 값이 MRR입니다.



**볼 곳**　두 번째 후보에 처음으로 기대 근거가 나타납니다. 찾은 비율과 첫 정답의 순위를 따로 읽어 보세요.

**애니메이션의 의미**　첫 정답이 2위이므로 이 질문의 RR은 0.5입니다.


### 셀 01 · 상위 두 개에서 기대 원문 찾기

대출 질문의 기대 원문은 library-1입니다. 목록 맨 앞에는 휴관 문서가 잘못 올라왔습니다.

In [1]:
ranked = ['library-2', 'library-1', 'library-3']
gold = {'library-1'}
top = ranked[:2]
recall = len(set(top) & gold) / len(gold)
print(recall)

1.0


1.0은 기대 원문 한 개를 두 문서 안에서 찾았다는 뜻입니다. 대출 문서가 첫 번째였다는 뜻은 아닙니다.

### 과제 02 · 첫 적중 순위의 역수 구하기

위 상위 두 건에서 첫 기대 원문의 순위를 1부터 세고 역수를 출력하세요.

종이에 1위와 2위를 먼저 쓰세요. 기대 문서 library-1은 2위에서 처음 등장하므로 역순위는 1÷2입니다. 코드는 첫 적중 순위를 찾은 뒤 그 역수를 구하면 됩니다.

In [2]:
# enumerate(top, start=1)로 순위를 찾으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>first_hit = next((rank for rank, doc in enumerate(top, start=1) if doc in gold), None)
print(1 / first_hit if first_hit else 0.0)</code></pre>
<p>실행 결과</p><pre><code>0.5</code></pre>
<p>두 번째에 처음 맞았으므로 0.5입니다. 이 한 문항의 RR을 다른 문항과 합치기 전에 k와 평가 대상을 같게 정하세요.</p>
</details>

### 셀 03 · 같은 k에서 두 질문의 평균 내기

앞에서 구한 대출 질문의 RR은 0.5입니다. 두 번째 휴관 질문도 상위 두 건에서 첫 적중을 확인한 뒤 두 문항의 평균을 냅니다.

In [3]:
from statistics import mean
holiday_top = ['library-2', 'library-1']
holiday_rank = holiday_top.index('library-2') + 1
print(mean([0.5, 1 / holiday_rank]))

0.75


0.75는 이 두 답변 가능 문항의 MRR@2입니다. 두 번째 문항에서는 첫 번째 순위에 정답이 와서 RR이 1이었습니다. 전체 검색기의 일반 성능은 이 두 문항만으로 알 수 없습니다.

## ID가 맞을 때도 문장을 다시 읽기

주차 질문처럼 기대 근거가 빈 경우 Recall의 분모는 0입니다. 이때 임의로 1점을 주지 않고 응답 유보를 별도로 봅니다. 근거가 있는 질문의 답도 숫자나 단위를 틀릴 수 있습니다.



**볼 곳**　‘답변 주장’를 기준으로 양쪽의 ‘인용 ID’와 ‘원문 내용’를 대조하세요.

**애니메이션의 의미**　출처가 맞더라도 답이 원문의 수치와 같은지 읽어야 합니다.


### 셀 04 · 유효한 인용이 붙은 잘못된 답

교육용 오류 문장에서 인용 ID가 기대 근거에 있는지 확인하고 원문과 숫자를 읽어 보세요.

In [4]:
record = {'id': 'library-1',
 'text': '도서 대출 기간은 14일입니다.',
 'source': '가상 도서관 이용 안내',
 'page': 1}
answer = '대출 기간은 999일입니다. [library-1]'
print(f"[{record['id']}]" in answer)
print(record['text'])

True
도서 대출 기간은 14일입니다.


True 다음의 원문에는 14일이라고 적혀 있습니다. 검색 적중과 인용 형식은 맞지만 답변 내용은 틀렸습니다. 자동 판정이나 모델 채점이 무엇을 검사했는지 기준을 남기세요.

## 돌아보기

첫 검색 순위를 낮추는 오류와 답변 숫자를 바꾸는 오류를 서로 다른 문제로 기록해 보세요. 어느 쪽을 고쳐야 하는지 지표 하나만으로는 정할 수 없습니다.